In [69]:
# run_spark_local_excel.py
import os
import sys
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1) Initialize PySpark (optional if pyspark is already on PATH)
findspark.init()

from pyspark.sql import SparkSession

# ---- Adjust these ----
BASE = "/home/henryx/urban/snp-dwh/spark_jobs"
LOCAL_JARS = ",".join([
    f"{BASE}/jars/spark-excel_2.12-0.13.7.jar",
    f"{BASE}/jars/poi-4.1.2.jar",
    f"{BASE}/jars/poi-ooxml-4.1.2.jar",
    f"{BASE}/jars/xmlbeans-3.1.0.jar",
    f"{BASE}/jars/commons-math3-3.6.1.jar",
    f"{BASE}/jars/ooxml-schemas-1.4.jar",
    f"{BASE}/jars/postgresql-42.7.3.jar",
])

# Use the real, exact file name (watch out: in your text there’s a stray space after the dash!)
# BAD: '...- 12-06-2025_vcf.xlsx'
# GOOD: '...-12-06-2025_vcf.xlsx'
local_excel_path = "/home/henryx/urban/snp-dwh/xscript_local/visualizador/datos_pnd2425- 12-06-2025_vcf.xlsx"

# If you truly want HDFS instead, use a proper HDFS URL and make sure Hadoop client configs are visible to Spark:
# hdfs_excel_path = "hdfs://172.18.21.152:8020/data/datos_pnd2425-12-06-2025_vcf_l1.xlsx"

app_name = "ExcelReaderLocal"

# 2) Build SparkSession for local use  

spark = (
    SparkSession.builder
    .appName("SimplePySparkDataFrame")
    .master("local[*]")  # local mode
    .config("spark.jars", LOCAL_JARS)
    .config("spark.driver.memory", "4g")       # 4 GB for driver
    .config("spark.executor.memory", "4g")     # 4 GB for executors
    .config("spark.sql.shuffle.partitions", "8")  # optional: fewer partitions for local
    .getOrCreate()
    )
  
df = (
    spark.read.format("com.crealytics.spark.excel")
    .option("header", "true")
    .option("inferSchema", "false") 
    .option("dataAddress", "'Indicadores PND24-25'!A3")
    .load(local_excel_path))

# 4) Quick checks
df.printSchema()
df.show(10, truncate=False)



root
 |-- codigo_: string (nullable = true)
 |-- TIPO: string (nullable = true)
 |-- EJE: string (nullable = true)
 |-- NOMBRE_DEL_OBJETIVO: string (nullable = true)
 |-- NOMBRE_DE_LA_POLITICA: string (nullable = true)
 |-- META: string (nullable = true)
 |-- INDICADOR: string (nullable = true)
 |-- FUENTE_DE_INFORMACION: string (nullable = true)
 |-- GRUPO_DE_DESAGREGACION: string (nullable = true)
 |-- NIVEL_DE_DESAGREGACION: string (nullable = true)
 |-- CODIGO_GEOGRAFICO_DPA: string (nullable = true)
 |-- MES_ANIO: string (nullable = true)
 |-- FECHA: string (nullable = true)
 |-- ESTIMADOR: string (nullable = true)
 |-- ERROR_ESTANDAR: string (nullable = true)
 |-- LIMITE_INFERIOR: string (nullable = true)
 |-- LIMITE_SUPERIOR: string (nullable = true)
 |-- COEFICIENTE_DE_VARIACION: string (nullable = true)
 |-- NUMERADOR: string (nullable = true)
 |-- DENOMINADOR: string (nullable = true)
 |-- NOMBRE_DEL_EJE: string (nullable = true)
 |-- PERIODICIDAD FICHA METODOLÓGICA: string (

In [70]:
from pyspark.sql import functions as F

if "EJE" in df.columns:
    df = df.withColumn("EJE", F.upper(F.col("EJE")))

In [71]:
df.groupBy("EJE").count().orderBy("count", ascending=False).show()


+--------------------+-----+
|                 EJE|count|
+--------------------+-----+
|              SOCIAL|18077|
|INFRAESTRUCTURA, ...| 4973|
|       INSTITUCIONAL| 2278|
|DESARROLLO ECONÓMICO| 2190|
|  GESTIÓN DE RIESGOS| 1196|
+--------------------+-----+



In [72]:
from pyspark.sql import functions as F

# -----------------------------
# 1) Remove numeric prefixes
#    a) First, the "1. text" style on two columns
# -----------------------------
prefix_cols_step1 = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA']
for c in prefix_cols_step1:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.trim(
                F.regexp_replace(F.col(c), r'^\s*\d+\.\s*', '')
            )
        )


In [73]:
df.groupBy("NOMBRE_DE_LA_POLITICA").count().orderBy("count", ascending=False).show()


+---------------------+-----+
|NOMBRE_DE_LA_POLITICA|count|
+---------------------+-----+
| 1 Prever, preveni...| 8318|
| 2 Optimizar las i...| 3023|
| 1 Garantizar el a...| 2537|
|                     | 1776|
| 2 Promover una ed...| 1207|
| 10 Impulsar la re...| 1196|
| 4 Conservar y res...| 1090|
| 5 Garantizar el a...| 1002|
| 3 Mejorar la pres...|  889|
| 4 Fortalecer la v...|  825|
| 1 Fomentar las op...|  792|
| 6 Fortalecer la r...|  598|
| 1 Fortalecer el S...|  598|
| 1. Fortalecer el ...|  598|
| 3  Fortalecer el ...|  459|
| 5 Fomentar la inv...|  436|
| 5 Garantizar la i...|  432|
| 13 Incrementar la...|  403|
| 8 Fortalecer la s...|  328|
| 2 Impulsar el Gob...|  325|
+---------------------+-----+
only showing top 20 rows



In [74]:
# -----------------------------
#    b) Then, the broader "1", "1.", "1  " variants on key cols
# -----------------------------
prefix_cols_step2 = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA', 'META', 'INDICADOR']
for c in prefix_cols_step2:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.trim(
                F.regexp_replace(F.col(c), r'^\s*\d+[\.\s]*', '')
            )
        )

In [75]:
df.groupBy("META").count().orderBy("count", ascending=False).show()


+--------------------+-----+
|                META|count|
+--------------------+-----+
|Reducir la tasa d...| 8318|
|Reducir la tasa d...| 2806|
|Mantener el índic...| 1776|
|Incrementar el po...| 1187|
|Mantener la propo...| 1082|
|Incrementar el po...|  950|
|Incrementar el ín...|  598|
|Incrementar el ín...|  598|
|Incrementar el ín...|  598|
|1.2 Mantener la c...|  598|
|1.1 Incrementar e...|  598|
|Reducir la tasa d...|  561|
|Incrementar los a...|  430|
|Incrementar la ta...|  406|
|Incrementar la ta...|  406|
|Incrementar la ta...|  406|
|Reducir la tasa d...|  403|
|Reducir la tasa e...|  367|
|Reducir la tasa e...|  360|
|Reducir la tasa d...|  328|
+--------------------+-----+
only showing top 20 rows



In [76]:
from pyspark.sql import functions as F

df = (
    df.withColumn("ESTIMADOR", F.col("ESTIMADOR").cast("double"))
      .withColumn("ERROR_ESTANDAR", F.col("ERROR_ESTANDAR").cast("double"))
      .withColumn("LIMITE_INFERIOR", F.col("LIMITE_INFERIOR").cast("double"))
      .withColumn("LIMITE_SUPERIOR", F.col("LIMITE_SUPERIOR").cast("double"))
      .withColumn("COEFICIENTE_DE_VARIACION", F.col("COEFICIENTE_DE_VARIACION").cast("double"))
      .withColumn("DENOMINADOR", F.col("DENOMINADOR").cast("double"))
)

In [77]:
df.count()

28714

In [78]:


"""
# -----------------------------
# 2) Coerce numeric columns (errors="coerce")
#    Invalid tokens -> null, then cast to double
# -----------------------------
numeric_cols = [
    "ESTIMADOR",
    "ERROR_ESTANDAR",
    "LIMITE_INFERIOR",
    "LIMITE_SUPERIOR",
    "COEFICIENTE_DE_VARIACION",
    "DENOMINADOR",
]

# Tokens that should become null (like your clean_numeric_val)
invalid_tokens = {'', '-', 'NaN', 'nan', '(omitted)'}

for c in numeric_cols:
    if c in df.columns:
        s = F.trim(F.col(c).cast("string"))
        # Optional: drop thousand separators or spaces before casting
        # s = F.regexp_replace(s, r'[,\s]', '')
        df = df.withColumn(
            c,
            F.when(s.isNull() | s.isin(list(invalid_tokens)) | (s == ''), None)
             .otherwise(s)
             .cast("double")   # invalid parses automatically become null (errors='coerce')
        )
# Round all float columns to 3 decimal places
for c, dtype in df.dtypes:
    if dtype in ("float", "double", "float64"):
        df = df.withColumn(c, F.round(F.col(c), 3))
        """

'\n# -----------------------------\n# 2) Coerce numeric columns (errors="coerce")\n#    Invalid tokens -> null, then cast to double\n# -----------------------------\nnumeric_cols = [\n    "ESTIMADOR",\n    "ERROR_ESTANDAR",\n    "LIMITE_INFERIOR",\n    "LIMITE_SUPERIOR",\n    "COEFICIENTE_DE_VARIACION",\n    "DENOMINADOR",\n]\n\n# Tokens that should become null (like your clean_numeric_val)\ninvalid_tokens = {\'\', \'-\', \'NaN\', \'nan\', \'(omitted)\'}\n\nfor c in numeric_cols:\n    if c in df.columns:\n        s = F.trim(F.col(c).cast("string"))\n        # Optional: drop thousand separators or spaces before casting\n        # s = F.regexp_replace(s, r\'[,\\s]\', \'\')\n        df = df.withColumn(\n            c,\n            F.when(s.isNull() | s.isin(list(invalid_tokens)) | (s == \'\'), None)\n             .otherwise(s)\n             .cast("double")   # invalid parses automatically become null (errors=\'coerce\')\n        )\n# Round all float columns to 3 decimal places\nfor c, dty

In [79]:
df.groupBy("ESTIMADOR").count().agg(F.sum("count")).show()


+----------+
|sum(count)|
+----------+
|     28714|
+----------+



In [80]:
df.select("ESTIMADOR").count()


28714

In [81]:
cols = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA', 'META', 'FUENTE_DE_INFORMACION']

for c in cols:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(
                F.col(c).isNull() | F.trim(F.col(c)).endswith('.'),
                F.trim(F.col(c))
            ).otherwise(
                F.concat(F.trim(F.col(c)), F.lit('.'))
            )
        )

In [82]:
df.groupBy("NOMBRE_DE_LA_POLITICA").count().agg(F.sum("count")).show()


+----------+
|sum(count)|
+----------+
|     28714|
+----------+



In [83]:
from pyspark.sql import functions as F

string_cols = [
    'codigo_',
    'TIPO',
    'EJE',
    'NOMBRE_DEL_OBJETIVO',
    'NOMBRE_DE_LA_POLITICA',
    'META',
    'INDICADOR',
    'FUENTE_DE_INFORMACION',
    'GRUPO_DE_DESAGREGACION',
    'NIVEL_DE_DESAGREGACION',
    'CODIGO_GEOGRAFICO_DPA',
    'MES_ANIO',
    'NOMBRE_DEL_EJE',
    'PERIODICIDAD FICHA METODOLÓGICA',
    'FECHA DE TRANSFERENCIA FICHA METODOLÓGICA',
    'DESAGREGACIÓN FICHA METODOLÓGICA ',
    'PERIODICIDAD DEL DATO',
]

for c in string_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("string"))

In [84]:
#df.show(5)

In [85]:
# === JDBC connection ===
usuario = "postgres"
password = "JHEQWR2ZUASDasdgASd98x"
host = "155.138.253.162"
puerto = "5432"
base_datos = "test_snp_pnd"

jdbc_url = f"jdbc:postgresql://{host}:{puerto}/{base_datos}"
jdbc_props = {
    "user": usuario,
    "password": password,
    "driver": "org.postgresql.Driver",
    # performance knobs:
    "stringtype": "unspecified",
    "rewriteBatchedStatements": "true",
    "tcpKeepAlive": "true",
}

In [86]:
from pyspark.sql import functions as F

def read_pg_table(spark, table, predicates=None):
    reader = (spark.read
              .format("jdbc")
              .option("url", jdbc_url)
              .option("dbtable", table)
              .option("numPartitions", 4)
              .option("fetchsize", 5000)
              .options(**jdbc_props))
    if predicates:
        return reader.option("predicates", predicates).load()
    return reader.load()

def write_append(df, table):
    (df.write
       .format("jdbc")
       .option("url", jdbc_url)
       .option("dbtable", table)
       .option("batchsize", 5000)
       .option("isolationLevel", "READ_COMMITTED")
       .options(**jdbc_props)
       .mode("append")
       .save())

In [136]:
df.groupBy('codigo_').count().show()

+-------+-----+
|codigo_|count|
+-------+-----+
|  1.5.2|  360|
|  1.1.1|   41|
|  1.3.1|  208|
|  1.8.1|   42|
|  1.4.1|  264|
|  1.5.1|  275|
|  1.7.1|  123|
|  1.2.1|   16|
|  1.3.2|  208|
|  1.4.2|  561|
|  1.3.3|  208|
|  1.3.4|   14|
|  1.3.5|  251|
|  1.5.3|  367|
|  2.2.1|   20|
|  2.5.1|  430|
|  2.1.5|  950|
|  2.7.1|   30|
|  2.1.4|  176|
|  2.3.3|  176|
+-------+-----+
only showing top 20 rows



In [133]:
# split hierarchical code once
parts = F.split(F.col("codigo_"), "\\.")
df = df.withColumn("id_objetivo", F.concat(parts.getItem(0), F.lit("."))) \
       .withColumn("id_politica", F.concat_ws(".", parts.getItem(0), parts.getItem(1))) \
       .withColumn("id_meta",     F.col("codigo_")) \
       .withColumn("id_indicador", F.concat(F.lit("ind_"), F.col("codigo_")))

# Dates
df = df.withColumn("fecha_dt", F.to_date("FECHA"))  # if already date-like, this is idempotent

"""
# Optional: trim key strings used in joins
for c in ["EJE", "NOMBRE_DEL_OBJETIVO", "NOMBRE_DE_LA_POLITICA",
          "META", "INDICADOR", "FUENTE_DE_INFORMACION",
          "GRUPO_DE_DESAGREGACION", "NIVEL_DE_DESAGREGACION",
          "CODIGO_GEOGRAFICO_DPA", "MES_ANIO"]:
    if c in df.columns:
        df = df.withColumn(c, F.trim(F.col(c)))"""

'\n# Optional: trim key strings used in joins\nfor c in ["EJE", "NOMBRE_DEL_OBJETIVO", "NOMBRE_DE_LA_POLITICA",\n          "META", "INDICADOR", "FUENTE_DE_INFORMACION",\n          "GRUPO_DE_DESAGREGACION", "NIVEL_DE_DESAGREGACION",\n          "CODIGO_GEOGRAFICO_DPA", "MES_ANIO"]:\n    if c in df.columns:\n        df = df.withColumn(c, F.trim(F.col(c)))'

In [88]:
# candidates from source
eje_new = (df
    .select(F.col("EJE").alias("nombre_eje"))
    .where(F.col("nombre_eje").isNotNull())
    .distinct())

# existing from Postgres
eje_db = read_pg_table(spark, "public.eje").select("id", "nombre_eje")

# only new rows
eje_to_insert = eje_new.join(eje_db.select("nombre_eje"), ["nombre_eje"], "left_anti")

if eje_to_insert.head(1):
    write_append(eje_to_insert, "public.eje")

# reload with ids
eje_full = read_pg_table(spark, "public.eje").select("id", "nombre_eje")
eje_full = eje_full.withColumn('id_eje', F.col('id'))

In [89]:
eje_full.show()

+---+--------------------+------+
| id|          nombre_eje|id_eje|
+---+--------------------+------+
|  1|              SOCIAL|     1|
|  2|DESARROLLO ECONÓMICO|     2|
|  3|INFRAESTRUCTURA, ...|     3|
|  4|       INSTITUCIONAL|     4|
|  5|  GESTIÓN DE RIESGOS|     5|
+---+--------------------+------+



In [90]:
obj_new = (df
    .select("id_objetivo",
            F.col("NOMBRE_DEL_OBJETIVO").alias("nombre_objetivo"),
            F.col("EJE").alias("nombre_eje"))
    .where(F.col("id_objetivo").isNotNull() & F.col("nombre_objetivo").isNotNull())
    .dropDuplicates(["id_objetivo"]))

obj_new = (obj_new
    .join(eje_full, "nombre_eje", "left")
    .select("id_objetivo", "nombre_objetivo", "id_eje"))

obj_db = read_pg_table(spark, "public.objetivo").select("id_objetivo")

obj_to_insert = obj_new.join(obj_db, ["id_objetivo"], "left_anti")

if obj_to_insert.head(1):
    write_append(obj_to_insert, "public.objetivo")

objetivo_full = read_pg_table(spark, "public.objetivo").select("id_objetivo", 'nombre_objetivo',"id_eje")

In [91]:
pol_new = (df
    .select("id_politica",
            F.col("NOMBRE_DE_LA_POLITICA").alias("nombre_politica"),
            "id_objetivo")
    .where(F.col("id_politica").isNotNull() & F.col("nombre_politica").isNotNull())
    .dropDuplicates(["id_politica"]))

# ensure FK present (sanity join)
pol_new = pol_new.join(objetivo_full.select("id_objetivo"), ["id_objetivo"], "inner")

pol_db = read_pg_table(spark, "public.politica").select("id_politica")
pol_to_insert = pol_new.join(pol_db, ["id_politica"], "left_anti")

if pol_to_insert.head(1):
    write_append(pol_to_insert, "public.politica")

politica_full = read_pg_table(spark, "public.politica").select("id_politica", 'nombre_politica',"id_objetivo")


In [92]:
print(politica_full.count() )

58


In [93]:
politica_full.show()


+-----------+--------------------+-----------+
|id_politica|     nombre_politica|id_objetivo|
+-----------+--------------------+-----------+
|        1.1|Contribuir a la r...|         1.|
|        1.2|Garantizar la inc...|         1.|
|        1.7|Implementar progr...|         1.|
|        1.3|Mejorar la presta...|         1.|
|        1.4|Fortalecer la vig...|         1.|
|        1.5|Garantizar el acc...|         1.|
|        1.8|Garantizar el der...|         1.|
|        2.7|Impulsar la creac...|         2.|
|        2.1|Garantizar el acc...|         2.|
|        2.2|Promover una educ...|         2.|
|        2.5|Fomentar la inves...|         2.|
|        2.8|Garantizar la pre...|         2.|
|        2.3|Fortalecer el sis...|         2.|
|        2.4|Desarrollar el si...|         2.|
|        3.1|Prever, prevenir ...|         3.|
|       3.10|Impulsar la reduc...|         3.|
|       3.17|Promover una educ...|         3.|
|       3.12|Contribuir al for...|         3.|
|       3.13|

In [94]:
meta_new = (df
    .select(F.col("id_meta").alias("id_meta"),
            F.col("META").alias("descripcion_meta"),
            "id_politica")
    .where(F.col("id_meta").isNotNull() & F.col("descripcion_meta").isNotNull())
    .dropDuplicates(["id_meta"]))

# sanity FK
meta_new = meta_new.join(politica_full.select("id_politica"), ["id_politica"], "inner")

meta_db = read_pg_table(spark, "public.meta").select("id_meta")
meta_to_insert = meta_new.join(meta_db, ["id_meta"], "left_anti")

if meta_to_insert.head(1):
    write_append(meta_to_insert, "public.meta")

meta_full = read_pg_table(spark, "public.meta").select("id_meta", 'descripcion_meta',"id_politica")


In [95]:
meta_full.count()

107

In [96]:
ind_new = (df
    .select(F.col("id_indicador").alias("id_indicador"),
            F.col("INDICADOR").alias("nombre_indicador"),
            F.col("id_meta").alias("id_meta"))
    .where(F.col("id_indicador").isNotNull() & F.col("nombre_indicador").isNotNull())
    .dropDuplicates(["id_indicador"]))

ind_new = ind_new.join(meta_full.select("id_meta"), ["id_meta"], "inner")

ind_db = read_pg_table(spark, "public.indicador").select("id_indicador")
ind_to_insert = ind_new.join(ind_db, ["id_indicador"], "left_anti")

if ind_to_insert.head(1):
    write_append(ind_to_insert, "public.indicador")

indicador_full = read_pg_table(spark, "public.indicador").select("id_indicador", 'nombre_indicador',"id_meta")


In [97]:
indicador_full.show(10)
indicador_full.count()

+------------+--------------------+-------+
|id_indicador|    nombre_indicador|id_meta|
+------------+--------------------+-------+
|   ind_1.1.1|Tasa de pobreza e...|  1.1.1|
|   ind_1.2.1|Tasa de pobreza p...|  1.2.1|
|   ind_1.7.1|Prevalencia de de...|  1.7.1|
|   ind_1.3.1|Cobertura de vacu...|  1.3.1|
|   ind_1.3.2|Cobertura de vacu...|  1.3.2|
|   ind_1.3.3|Cobertura de vacu...|  1.3.3|
|   ind_1.3.4|Gasto de bolsillo...|  1.3.4|
|   ind_1.3.5|Tasa de médicos f...|  1.3.5|
|   ind_1.4.1|Porcentaje de per...|  1.4.1|
|   ind_1.4.2|Tasa de mortalida...|  1.4.2|
+------------+--------------------+-------+
only showing top 10 rows



107

In [98]:
fuente_new = (df
    .select(F.col("FUENTE_DE_INFORMACION").alias("nombre_fuente"))
    .where(F.col("nombre_fuente").isNotNull())
    .distinct())

fuente_db = read_pg_table(spark, "public.fuente_informacion").select("id", "nombre_fuente")

fuente_to_insert = fuente_new.join(fuente_db.select("nombre_fuente"), ["nombre_fuente"], "left_anti")

if fuente_to_insert.head(1):
    write_append(fuente_to_insert, "public.fuente_informacion")

fuente_full = read_pg_table(spark, "public.fuente_informacion").select("id", "nombre_fuente")

In [99]:
df_filtered = (
    df.select("GRUPO_DE_DESAGREGACION", "NIVEL_DE_DESAGREGACION", "CODIGO_GEOGRAFICO_DPA")
      .dropna(subset=["GRUPO_DE_DESAGREGACION", "NIVEL_DE_DESAGREGACION"])
      .dropDuplicates()
)

In [100]:
df.groupBy("GRUPO_DE_DESAGREGACION").count().orderBy("count", ascending=False).show(50)


+----------------------+-----+
|GRUPO_DE_DESAGREGACION|count|
+----------------------+-----+
|              Cantonal|15014|
|            Provincial| 6258|
|            Geográfico| 1245|
|             Provincia| 1083|
|        Grupos etarios|  840|
|             Municipal|  563|
|                  Sexo|  506|
|         Otros ámbitos|  413|
|                  Área|  408|
|  Zonas de Planific...|  396|
|           Provincial |  288|
|          Tipo de arma|  255|
|        Grupos de Edad|  242|
|  Provincia de resi...|  192|
|  Provincia sede ma...|  162|
|         Base indexada|  159|
|                 Etnia|  117|
|  Área de conocimiento|  100|
|        Grupos de edad|   96|
|          Discapacidad|   56|
|             Provicial|   48|
|  Grado de intensid...|   48|
|  Demarcación Hidro...|   35|
|               Deporte|   32|
|  Tipo de financiam...|   21|
|                 Sexo |   18|
|     Ámbito de fomento|   16|
|        Otros ámbitos |   15|
|  Fuente de financi...|   14|
|       

In [101]:
desag_new = (df
    .select(F.col("GRUPO_DE_DESAGREGACION").alias("grupo_desagregacion"),
            F.col("NIVEL_DE_DESAGREGACION").alias("nivel_desagregacion"),
            F.col("CODIGO_GEOGRAFICO_DPA").cast("string").alias("codigo_geografico_dpa")) 
    .dropDuplicates(["grupo_desagregacion", "nivel_desagregacion", "codigo_geografico_dpa"]))

In [102]:
desag_new.count()

1060

In [103]:
print(eje_full.count() )
print(objetivo_full.count() )
print(politica_full.count() )
print(meta_full.count() )
print(indicador_full.count() )
print(fuente_full.count() )


5


10


58


107


107
92


In [104]:
desag_new = (df
    .select(F.col("GRUPO_DE_DESAGREGACION").alias("grupo_desagregacion"),
            F.col("NIVEL_DE_DESAGREGACION").alias("nivel_desagregacion"),
            F.col("CODIGO_GEOGRAFICO_DPA").cast("string").alias("codigo_geografico_dpa"))
    .where(F.col("grupo_desagregacion").isNotNull() & F.col("nivel_desagregacion").isNotNull())
    .dropDuplicates(["grupo_desagregacion", "nivel_desagregacion", "codigo_geografico_dpa"]))

desag_db = read_pg_table(
    spark, "public.desagregacion"
).select("id_desagregacion", "grupo_desagregacion", "nivel_desagregacion", "codigo_geografico_dpa")

desag_to_insert = desag_new.join(
    desag_db.select("grupo_desagregacion","nivel_desagregacion","codigo_geografico_dpa"),
    ["grupo_desagregacion","nivel_desagregacion","codigo_geografico_dpa"],
    "left_anti"
)

if desag_to_insert.head(1):
    write_append(desag_to_insert, "public.desagregacion")

desag_full = read_pg_table(
    spark, "public.desagregacion"
).select("id_desagregacion", "grupo_desagregacion", "nivel_desagregacion", "codigo_geografico_dpa")


In [105]:
desag_new.count()

1060

In [106]:
desag_new.count()

1060

In [107]:
df.count()

28714

In [108]:
desag_new.count()

1060

In [109]:
desag_full = desag_full.withColumn('codigo_geografico_de_dpa', F.col('codigo_geografico_dpa'))

In [110]:
tiempo_new = (df
    .select(
        F.col("MES_ANIO").alias("mes_anio"),
        F.col("fecha_dt").alias("fecha"),
        F.col("PERIODICIDAD FICHA METODOLÓGICA").alias("periodicidad_ficha_metodologica"),
        F.col("FECHA DE TRANSFERENCIA FICHA METODOLÓGICA").cast("string").alias("fecha_transferencia_ficha_metodologica"),
        F.col("PERIODICIDAD DEL DATO").alias("periodicidad_dato"),
    )
    .where(F.col("mes_anio").isNotNull())
    .dropDuplicates(["mes_anio","fecha","periodicidad_ficha_metodologica",
                     "fecha_transferencia_ficha_metodologica","periodicidad_dato"])
)

tiempo_db = read_pg_table(
    spark, "public.tiempo"
).select("id_tiempo","mes_anio","fecha","periodicidad_ficha_metodologica",
         "fecha_transferencia_ficha_metodologica","periodicidad_dato")

tiempo_to_insert = tiempo_new.join(
    tiempo_db.drop("id_tiempo"),
    ["mes_anio","fecha","periodicidad_ficha_metodologica",
     "fecha_transferencia_ficha_metodologica","periodicidad_dato"],
    "left_anti"
)

if tiempo_to_insert.head(1):
    write_append(tiempo_to_insert, "public.tiempo")

tiempo_full = read_pg_table(
    spark, "public.tiempo"
).select("id_tiempo","mes_anio","fecha","periodicidad_ficha_metodologica",
         "fecha_transferencia_ficha_metodologica","periodicidad_dato")


In [132]:
print(eje_full.count() )
print(objetivo_full.count() )
print(politica_full.count() )
print(meta_full.count() )
print(indicador_full.count() )
print(fuente_full.count() )
print(desag_full.count() )
print(tiempo_full.count() )

5


10


58


107


107


92


1060
1132


In [112]:
tiempo_new.count()

1132

In [113]:
indicador_full.printSchema()

root
 |-- id_indicador: string (nullable = true)
 |-- nombre_indicador: string (nullable = true)
 |-- id_meta: string (nullable = true)



In [114]:
fuente_full = fuente_full.withColumn('id_fuente', F.col('id'))

In [115]:
df.select('NOMBRE_DE_LA_POLITICA')

DataFrame[NOMBRE_DE_LA_POLITICA: string]

In [116]:
# bring FK ids in via joins
fact = (df.alias("df")
    # indicador
    .join(indicador_full.alias("indicador"), on= "id_indicador" , how="inner")
    # desagregacion
    .join(desag_full.alias("desag"),
          (F.col("df.GRUPO_DE_DESAGREGACION") == F.col("desag.grupo_desagregacion")) &
          (F.col("df.NIVEL_DE_DESAGREGACION") == F.col("desag.nivel_desagregacion")) &
          (F.col("df.CODIGO_GEOGRAFICO_DPA").cast("string") == F.col("desag.codigo_geografico_dpa")),
          "left")
    # tiempo
    .join(tiempo_full.alias("tiempo"),
          (F.col("df.MES_ANIO") == F.col("tiempo.mes_anio")) &
          (F.col("df.fecha_dt") == F.col("tiempo.fecha")) &
          (F.col("df.PERIODICIDAD FICHA METODOLÓGICA") == F.col("tiempo.periodicidad_ficha_metodologica")) &
          (F.col("df.FECHA DE TRANSFERENCIA FICHA METODOLÓGICA").cast("string") == F.col("tiempo.fecha_transferencia_ficha_metodologica")) &
          (F.col("df.PERIODICIDAD DEL DATO") == F.col("tiempo.periodicidad_dato")),
          "left")
    # fuente
    .join(fuente_full.alias("fuente"), F.col("df.FUENTE_DE_INFORMACION") == F.col("fuente.nombre_fuente"), "left")
)



In [117]:
fact.count()

28714

In [118]:
# final projection to staging schema
fact_stg = (fact
    .select(
        F.col("id_indicador").alias("id_indicador"),
        F.col("id_desagregacion").alias("id_desagregacion"),
        F.col("id_tiempo").alias("id_tiempo"),
        F.col("id_fuente").alias("id_fuente"),
        F.col("ESTIMADOR").cast("double").alias("estimador"),
        F.col("ERROR_ESTANDAR").cast("double").alias("error_estandar"),
        F.col("LIMITE_INFERIOR").cast("double").alias("limite_inferior"),
        F.col("LIMITE_SUPERIOR").cast("double").alias("limite_superior"),
        F.col("COEFICIENTE_DE_VARIACION").cast("double").alias("coeficiente_variacion"),
        F.col("NUMERADOR").cast("double").alias("numerador"),
        F.col("DENOMINADOR").cast("double").alias("denominador"),
        F.col("NOMBRE_DEL_EJE").alias("nombre_del_eje"),
        F.col("TIPO").alias("tipo")
    )

    .repartition(8)  # tune per machine
)
#    .where(F.col("id_desagregacion").isNotNull() & F.col("id_tiempo").isNotNull())

fact_stg.select([
    'id_indicador', 
    'id_desagregacion', 
    'id_tiempo', 
    'id_fuente', 
    'estimador', 
    'error_estandar', 
    'limite_inferior', 
    'limite_superior', 
    'coeficiente_variacion', 
    'numerador', 
    'denominador', 
    'nombre_del_eje', 
    'tipo'
])

In [119]:
fact_stg.select([
    'id_indicador', 
    'id_desagregacion', 
    'id_tiempo', 
    'id_fuente', 
    'estimador', 
    'error_estandar', 
    'limite_inferior', 
    'limite_superior', 
    'coeficiente_variacion', 
    'numerador', 
    'denominador', 
    'nombre_del_eje', 
    'tipo'
])

DataFrame[id_indicador: string, id_desagregacion: int, id_tiempo: int, id_fuente: int, estimador: double, error_estandar: double, limite_inferior: double, limite_superior: double, coeficiente_variacion: double, numerador: double, denominador: double, nombre_del_eje: string, tipo: string]

In [120]:
fact_stg.count

<bound method DataFrame.count of DataFrame[id_indicador: string, id_desagregacion: int, id_tiempo: int, id_fuente: int, estimador: double, error_estandar: double, limite_inferior: double, limite_superior: double, coeficiente_variacion: double, numerador: double, denominador: double, nombre_del_eje: string, tipo: string]>

In [121]:
from pyspark.sql.functions import monotonically_increasing_id 
fact_stg_con_id = fact_stg.withColumn(
    "id", 
    monotonically_increasing_id()
)

In [122]:
fact_stg_con_id = fact_stg_con_id.withColumn("id", F.col("id").cast("int"))


In [123]:
fact_stg_con_id = fact_stg_con_id.select([
    'id',
    'id_indicador', 
    'id_desagregacion', 
    'id_tiempo', 
    'id_fuente', 
    'estimador', 
    'error_estandar', 
    'limite_inferior', 
    'limite_superior', 
    'coeficiente_variacion', 
    'numerador', 
    'denominador', 
    'nombre_del_eje', 
    'tipo'
])

In [125]:
fact_clean.count()

28714

In [127]:
fact_clean.show()

+------------+----------------+---------+---------+------------------+--------------+---------------+---------------+---------------------+-----------+-----------+--------------------+----+
|id_indicador|id_desagregacion|id_tiempo|id_fuente|         estimador|error_estandar|limite_inferior|limite_superior|coeficiente_variacion|  numerador|denominador|      nombre_del_eje|tipo|
+------------+----------------+---------+---------+------------------+--------------+---------------+---------------+---------------------+-----------+-----------+--------------------+----+
|   ind_3.1.1|              74|       34|       28|13.171759747102213|          NULL|           NULL|           NULL|                 NULL|        1.0|     7592.0|Tasa por 100.000 ...|   P|
|  ind_3.10.2|             125|       40|       29|       30.02005348|          NULL|           NULL|           NULL|                 NULL|       NULL|       NULL|        Adimensional|   N|
|   ind_3.1.1|              97|       35|       28

In [128]:
fact_clean = fact_clean.dropDuplicates(["id_indicador", "id_desagregacion", "id_tiempo"])

In [131]:
fact_clean.count()

24935

In [130]:
(
    fact_clean.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.mediciones")
    .option("batchsize", 5000)
    .option("isolationLevel", "READ_COMMITTED")
    .options(**jdbc_props)
    .mode("append")
    .save()
)

In [ ]:
# 11.1 write staging table
from pyspark.sql import functions as F

# 0) columns we will write (NO 'id'!)
cols_target = [
    "id_indicador", "id_desagregacion", "id_tiempo", "id_fuente",
    "estimador", "error_estandar", "limite_inferior", "limite_superior",
    "coeficiente_variacion", "numerador", "denominador",
    "nombre_del_eje", "tipo"
]

# 1) dedupe in Spark just in case
#fact_clean = fact_stg.select(*cols_target).dropDuplicates(
#    ["id_indicador", "id_desagregacion", "id_tiempo"]
#)
fact_clean = fact_stg.select(*cols_target)
# 2) read existing keys from DB
existing = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.mediciones")
    .option("numPartitions", 4)
    .option("fetchsize", 5000)
    .options(**jdbc_props)
    .load()
    .select("id_indicador", "id_desagregacion", "id_tiempo")
    .dropDuplicates()
)

# 3) keep only NEW rows via left_anti
to_insert = fact_clean.join(
    existing,
    on=["id_indicador", "id_desagregacion", "id_tiempo"],
    how="left_anti"
)

# 4) write append (no 'id' column, so PK auto-generates)
(
    fact_clean.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.mediciones")
    .option("batchsize", 5000)
    .option("isolationLevel", "READ_COMMITTED")
    .options(**jdbc_props)
    .mode("append")
    .save()
)


25/10/21 21:50:26 ERROR Executor: Exception in task 0.0 in stage 529.0 (TID 1332)
java.sql.BatchUpdateException: Batch entry 808 INSERT INTO public.mediciones ("id_indicador","id_desagregacion","id_tiempo","id_fuente","estimador","error_estandar","limite_inferior","limite_superior","coeficiente_variacion","numerador","denominador","nombre_del_eje","tipo") VALUES (('ind_3.13.1'),('389'::int4),('594'::int4),('32'::int4),('1.0'::double precision),(NULL),(NULL),(NULL),(NULL),('10.0'::double precision),('1525669.0'::double precision),('Tasa por cada 100.000 habitantes mujeres.'),('N')) was aborted: ERROR: duplicate key value violates unique constraint "uq_mediciones_ind_des_tiempo"
  Detail: Key (id_indicador, id_desagregacion, id_tiempo)=(ind_3.13.1, 389, 594) already exists.  Call getNextException to see other errors in the batch.
	at org.postgresql.jdbc.BatchResultHandler.handleError(BatchResultHandler.java:165)
	at org.postgresql.core.v3.QueryExecutorImpl.processResults(QueryExecutorImp

Py4JJavaError: An error occurred while calling o2347.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 529.0 failed 1 times, most recent failure: Lost task 0.0 in stage 529.0 (TID 1332) (10.255.255.254 executor driver): java.sql.BatchUpdateException: Batch entry 808 INSERT INTO public.mediciones ("id_indicador","id_desagregacion","id_tiempo","id_fuente","estimador","error_estandar","limite_inferior","limite_superior","coeficiente_variacion","numerador","denominador","nombre_del_eje","tipo") VALUES (('ind_3.13.1'),('389'::int4),('594'::int4),('32'::int4),('1.0'::double precision),(NULL),(NULL),(NULL),(NULL),('10.0'::double precision),('1525669.0'::double precision),('Tasa por cada 100.000 habitantes mujeres.'),('N')) was aborted: ERROR: duplicate key value violates unique constraint "uq_mediciones_ind_des_tiempo"
  Detail: Key (id_indicador, id_desagregacion, id_tiempo)=(ind_3.13.1, 389, 594) already exists.  Call getNextException to see other er

25/10/21 21:50:27 WARN TaskSetManager: Lost task 3.0 in stage 529.0 (TID 1335) (10.255.255.254 executor driver): TaskKilled (Stage cancelled: Job aborted due to stage failure: Task 0 in stage 529.0 failed 1 times, most recent failure: Lost task 0.0 in stage 529.0 (TID 1332) (10.255.255.254 executor driver): java.sql.BatchUpdateException: Batch entry 808 INSERT INTO public.mediciones ("id_indicador","id_desagregacion","id_tiempo","id_fuente","estimador","error_estandar","limite_inferior","limite_superior","coeficiente_variacion","numerador","denominador","nombre_del_eje","tipo") VALUES (('ind_3.13.1'),('389'::int4),('594'::int4),('32'::int4),('1.0'::double precision),(NULL),(NULL),(NULL),(NULL),('10.0'::double precision),('1525669.0'::double precision),('Tasa por cada 100.000 habitantes mujeres.'),('N')) was aborted: ERROR: duplicate key value violates unique constraint "uq_mediciones_ind_des_tiempo"
  Detail: Key (id_indicador, id_desagregacion, id_tiempo)=(ind_3.13.1, 389, 594) alread

In [ ]:

spark.stop()
